# 07 — Join & Feature Engineering

This notebook assembles the analytical dataset used for modeling.
The BTS on-time table is the spine; three sources join onto it.

| Join | Key | What it adds |
|------|-----|--------------|
| NOAA weather | Origin + FlightDate | Daily weather summary at departure airport |
| T-100 domestic | Carrier + Origin + Dest + Year + Month | Monthly route capacity & carrier share |
| DB1B fares | Carrier + Origin + Dest + Year + Quarter | Quarterly nonstop average fare |

Derived features (dep_hour_bucket, season, is_weekend, is_holiday_week, is_late)
are computed from existing columns — no additional joins.

**Output:** `data/processed/flights_featured.parquet`

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from config import RAW_DATA_PATH, PROCESSED_DATA_PATH

AIRPORTS = ['IAD', 'DCA', 'BWI']
AIRPORT_COLORS = {'IAD': '#1f77b4', 'DCA': '#ff7f0e', 'BWI': '#2ca02c'}

## Section 1 — Base Table

Load BTS on-time performance data. Cancelled flights are kept (flagged via `Cancelled == 1`);
delay columns will be NaN for those rows and can be excluded during model training.
Year, Quarter, and a date key are added to support downstream joins.

In [2]:
ot = pd.read_parquet(PROJECT_ROOT / PROCESSED_DATA_PATH / 'ontime_dmv.parquet')

ot['FlightDate'] = pd.to_datetime(ot['FlightDate'])
ot['Year']    = ot['FlightDate'].dt.year
ot['Quarter'] = ot['FlightDate'].dt.quarter
ot['date']    = ot['FlightDate'].dt.date

n_total  = len(ot)
n_fly    = int((ot['Cancelled'] == 0).sum())
n_cancel = int((ot['Cancelled'] == 1).sum())
print(f'Base table: {n_total:,} rows  x  {ot.shape[1]} columns')
print(f'  Operated:  {n_fly:,}   Cancelled: {n_cancel:,}')
print(f'  Date range: {ot["FlightDate"].min().date()} to {ot["FlightDate"].max().date()}')
print(f'  Origins: {sorted(ot["Origin"].unique())}')
print(f'  Carriers: {ot["Reporting_Airline"].nunique()} unique')

Base table: 2,823,044 rows  x  36 columns
  Operated:  2,754,055   Cancelled: 68,989
  Date range: 2015-01-01 to 2026-01-31
  Origins: ['BWI', 'DCA', 'IAD']
  Carriers: 18 unique


## Section 2 — NOAA Weather Join

NOAA is hourly (~24 rows per airport per day). We aggregate to one row per airport per
calendar date before joining — the daily minimum, maximum, and any-occurrence statistics
that characterise the full day at that airport.

The join is then many-to-one: all flights departing IAD on a given date receive the same
daily IAD weather summary. `Origin` (on-time) = `Airport` (NOAA) is valid because the
on-time data is pre-filtered to DMV origins — the only airports NOAA covers.

In [3]:
frames = []
for ap in AIRPORTS:
    path = PROJECT_ROOT / RAW_DATA_PATH / 'noaa' / f'noaa_{ap.lower()}.parquet'
    frames.append(pd.read_parquet(path))
noaa = pd.concat(frames, ignore_index=True)

noaa['DATE'] = pd.to_datetime(noaa['DATE'])
noaa['date'] = noaa['DATE'].dt.date
noaa['is_ts']  = noaa['HourlyPresentWeatherType'].str.contains('TS',          na=False)
noaa['is_sn']  = noaa['HourlyPresentWeatherType'].str.contains('SN|SG|IC|PL', na=False)
noaa['is_fg']  = noaa['HourlyPresentWeatherType'].str.contains('FG|BR',       na=False)
noaa['is_ifr'] = noaa['HourlyVisibility'] < 3

daily_noaa = (
    noaa.groupby(['Airport', 'date'])
    .agg(
        wx_had_ts     = ('is_ts',              'any'),
        wx_had_snow   = ('is_sn',              'any'),
        wx_had_fog    = ('is_fg',              'any'),
        wx_had_ifr    = ('is_ifr',             'any'),
        wx_min_vis    = ('HourlyVisibility',   'min'),
        wx_max_wind   = ('HourlyWindSpeed',    'max'),
        wx_total_prcp = ('HourlyPrecipitation','sum'),
    )
    .reset_index()
    .rename(columns={'Airport': 'Origin'})
)
del noaa

print(f'Daily NOAA: {len(daily_noaa):,} airport-day rows')
print(f'  Coverage: {daily_noaa["date"].min()} to {daily_noaa["date"].max()}')

before = len(ot)
ot = ot.merge(daily_noaa, on=['Origin', 'date'], how='left')
del daily_noaa

match_pct = ot['wx_min_vis'].notna().mean() * 100
print(f'After NOAA join: {len(ot):,} rows (delta: {len(ot)-before})  |  match rate: {match_pct:.1f}%')

Daily NOAA: 11,670 airport-day rows
  Coverage: 2015-01-01 to 2025-08-25
After NOAA join: 2,823,044 rows (delta: 0)  |  match rate: 95.5%


## Section 3 — T-100 Route Capacity Join

T-100 data is at carrier + origin + dest + aircraft-type grain. We aggregate across aircraft
types first (summing seats and departures) to get a clean carrier + route + month table,
then compute `t100_carrier_share` — that carrier's fraction of departures on the route
that month. This captures competitive context that the on-time file lacks entirely.

Join key: `Reporting_Airline + Origin + Dest + Year + Month` — all 18 on-time carrier codes
confirmed present in T-100 during the audit.

In [4]:
t100_raw = pd.read_parquet(PROJECT_ROOT / PROCESSED_DATA_PATH / 't100_domestic_dmv.parquet')

# Aggregate across aircraft types to carrier+route+month
t100 = (
    t100_raw
    .groupby(['UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'YEAR', 'MONTH'])
    .agg(
        t100_seats      = ('SEATS',               'sum'),
        t100_departures = ('DEPARTURES_PERFORMED', 'sum'),
        t100_passengers = ('PASSENGERS',           'sum'),
    )
    .reset_index()
)
del t100_raw

# Carrier share of departures on each route-month
route_deps = (
    t100.groupby(['ORIGIN', 'DEST', 'YEAR', 'MONTH'])['t100_departures']
    .sum()
    .rename('route_total_deps')
    .reset_index()
)
t100 = t100.merge(route_deps, on=['ORIGIN', 'DEST', 'YEAR', 'MONTH'])
t100['t100_carrier_share'] = (t100['t100_departures'] / t100['route_total_deps']).round(4)
t100.drop(columns=['route_total_deps'], inplace=True)

t100.rename(columns={
    'UNIQUE_CARRIER': 'Reporting_Airline',
    'ORIGIN':         'Origin',
    'DEST':           'Dest',
    'YEAR':           'Year',
    'MONTH':          'Month',
}, inplace=True)

print(f'T-100 join table: {len(t100):,} rows')
n_before_cols = ot.shape[1]
ot = ot.merge(t100, on=['Reporting_Airline', 'Origin', 'Dest', 'Year', 'Month'], how='left')
del t100

match_pct = ot['t100_seats'].notna().mean() * 100
new_cols  = ot.shape[1] - n_before_cols
print(f'After T-100 join: {len(ot):,} rows  |  match rate: {match_pct:.1f}%  |  {new_cols} new columns')
print()

# Inspect unmatched flights
unmatched = (
    ot[ot['t100_seats'].isna()]
    [['Reporting_Airline', 'Origin', 'Dest']]
    .value_counts()
    .head(10)
)
print('Top unmatched carrier+route combos (not in T-100 domestic):')
print(unmatched.to_string())

T-100 join table: 77,068 rows
After T-100 join: 2,823,044 rows  |  match rate: 100.0%  |  4 new columns

Top unmatched carrier+route combos (not in T-100 domestic):
Reporting_Airline  Origin  Dest
OH                 DCA     BGR     64
                           CLE     41
                           BNA     31
                           MCI     18
G4                 BWI     AVL     15
OH                 DCA     TYS     14
AA                 DCA     PHL     12
OH                 DCA     GSO     12
AA                 DCA     ATL     11
G4                 BWI     SAV     11


## Section 4 — DB1B Fare Join

DB1B is a 10% ticket sample at itinerary grain. We filter to:
- DMV-origin rows (`Origin` in IAD/DCA/BWI)
- Nonstop itineraries only (`MktCoupons == 1`) — connecting fares mix different products at different price points
- Positive fares (`MktFare > 0`) — zero-fare records are administrative

After aggregating to `RPCarrier + Origin + Dest + Year + Quarter`, the join key maps to
`Reporting_Airline + Origin + Dest + Year + Quarter` in on-time. All 18 on-time carrier codes
were audit-confirmed present in DB1B `RPCarrier`.

In [5]:
db1b = pd.read_parquet(
    PROJECT_ROOT / PROCESSED_DATA_PATH / 'db1b_dmv.parquet',
    columns=['Origin', 'Dest', 'Year', 'Quarter', 'RPCarrier', 'MktFare', 'MktCoupons', 'Passengers'],
)

db1b_ns = db1b[
    db1b['Origin'].isin(AIRPORTS) &
    (db1b['MktCoupons'] == 1) &
    (db1b['MktFare'] > 0)
].copy()
del db1b

db1b_agg = (
    db1b_ns
    .groupby(['RPCarrier', 'Origin', 'Dest', 'Year', 'Quarter'])
    .agg(
        db1b_avg_fare   = ('MktFare',    'mean'),
        db1b_pax_sample = ('Passengers', 'sum'),
    )
    .reset_index()
    .rename(columns={'RPCarrier': 'Reporting_Airline'})
)
db1b_agg['db1b_avg_fare'] = db1b_agg['db1b_avg_fare'].round(2)
del db1b_ns

avg_fare = db1b_agg['db1b_avg_fare'].mean()
print(f'DB1B aggregated: {len(db1b_agg):,} carrier+route+quarter rows')
print(f'  Mean nonstop fare across all routes: ${avg_fare:.0f}')

n_before_cols = ot.shape[1]
ot = ot.merge(db1b_agg, on=['Reporting_Airline', 'Origin', 'Dest', 'Year', 'Quarter'], how='left')
del db1b_agg

match_pct = ot['db1b_avg_fare'].notna().mean() * 100
new_cols  = ot.shape[1] - n_before_cols
print(f'After DB1B join: {len(ot):,} rows  |  match rate: {match_pct:.1f}%  |  {new_cols} new columns')

DB1B aggregated: 38,844 carrier+route+quarter rows
  Mean nonstop fare across all routes: $214
After DB1B join: 2,823,044 rows  |  match rate: 93.7%  |  2 new columns


## Section 6 — Derived Features

All computed from existing columns — no additional joins.

| Feature | Logic |
|---------|-------|
| `dep_hour_bucket` | CRSDepTime → Early (5-8) / Morning (9-11) / Afternoon (12-17) / Evening (18-21) / Night |
| `season` | Month → Winter / Spring / Summer / Fall |
| `is_weekend` | DayOfWeek in {6, 7} |
| `is_holiday_week` | Thanksgiving Nov 20-30, Christmas-NY Dec 20-Jan 2, July 4th Jun 30-Jul 6 |
| `is_late` | ArrDelayMinutes > 15 (BTS definition); NaN preserved for cancelled flights |

In [7]:
# Departure hour bucket (vectorised from HHMM integer)
dep_hour = ot['CRSDepTime'].fillna(0).astype(int) // 100
ot['dep_hour_bucket'] = np.select(
    [
        (dep_hour >= 5)  & (dep_hour < 9),
        (dep_hour >= 9)  & (dep_hour < 12),
        (dep_hour >= 12) & (dep_hour < 18),
        (dep_hour >= 18) & (dep_hour < 22),
    ],
    ['Early', 'Morning', 'Afternoon', 'Evening'],
    default='Night',
)

# Season
month_season = {
    12: 'Winter', 1: 'Winter',  2: 'Winter',
     3: 'Spring', 4: 'Spring',  5: 'Spring',
     6: 'Summer', 7: 'Summer',  8: 'Summer',
     9: 'Fall',  10: 'Fall',   11: 'Fall',
}
ot['season'] = ot['Month'].map(month_season)

# Weekend flag
ot['is_weekend'] = ot['DayOfWeek'].isin([6, 7]).astype(int)

# Holiday-week flag
md = ot['FlightDate'].dt.month * 100 + ot['FlightDate'].dt.day
ot['is_holiday_week'] = (
    ((md >= 1120) & (md <= 1130)) |
    ((md >= 1220) | (md <= 102))  |
    ((md >= 630)  & (md <= 706))
).astype(int)

# Late flag (NaN preserved for cancelled via Int8 nullable integer)
ot['is_late'] = (ot['ArrDelayMinutes'] > 15).astype('Int8')

print('Derived feature distributions:')
print('  dep_hour_bucket:', ot['dep_hour_bucket'].value_counts().sort_index().to_dict())
print('  season:         ', ot['season'].value_counts().sort_index().to_dict())
weekend_share = ot['is_weekend'].mean()
holiday_share = ot['is_holiday_week'].mean()
late_share    = ot.loc[ot['Cancelled'] == 0, 'is_late'].mean()
print(f'  is_weekend:      {weekend_share:.3f}')
print(f'  is_holiday_week: {holiday_share:.3f}')
print(f'  is_late (operated): {late_share:.3f}')

Derived feature distributions:
  dep_hour_bucket: {'Afternoon': 1055092, 'Early': 677914, 'Evening': 497662, 'Morning': 448581, 'Night': 143795}
  season:          {'Fall': 704222, 'Spring': 702621, 'Summer': 727148, 'Winter': 689053}
  is_weekend:      0.263
  is_holiday_week: 0.085
  is_late (operated): 0.190


## Section 7 — Validation & Null Audit

Check join coverage and feature completeness before saving.
Low match rates indicate key mismatches or date-range gaps worth investigating.

In [8]:
print(f'Final shape: {ot.shape[0]:,} rows  x  {ot.shape[1]} columns')
print()

print('=== Join Coverage ===')
for label, col in [
    ('NOAA weather',   'wx_min_vis'),
    ('T-100 capacity', 't100_seats'),
    ('DB1B fare',      'db1b_avg_fare'),
]:
    n_matched = int(ot[col].notna().sum())
    pct = n_matched / len(ot) * 100
    print(f'  {label:<18}: {n_matched:>10,}  ({pct:.1f}%)')

print()
print('=== Null Rates for Key Features ===')
feature_cols = [
    'wx_had_ts', 'wx_min_vis', 'wx_max_wind', 'wx_total_prcp',
    't100_seats', 't100_carrier_share',
    'db1b_avg_fare',
    'dep_hour_bucket', 'season', 'is_weekend', 'is_holiday_week',
    'ArrDelayMinutes', 'NASDelay', 'WeatherDelay', 'is_late',
]
null_df = pd.DataFrame({
    'Null Count': [int(ot[c].isna().sum())             for c in feature_cols],
    'Null %':     [round(ot[c].isna().mean() * 100, 1) for c in feature_cols],
}, index=feature_cols)
print(null_df.to_string())

print()
print('=== Delay Target (operated flights only) ===')
operated = ot[ot['Cancelled'] == 0]
ontime_pct = (operated['ArrDelayMinutes'] <= 15).mean() * 100
late_pct   = (operated['ArrDelayMinutes'] >  15).mean() * 100
mean_delay = operated['ArrDelayMinutes'].mean()
med_delay  = operated['ArrDelayMinutes'].median()
print(f'  On-time (<= 15 min): {ontime_pct:.1f}%')
print(f'  Late    (>  15 min): {late_pct:.1f}%')
print(f'  Mean arrival delay:  {mean_delay:.1f} min')
print(f'  Median arrival delay:{med_delay:.1f} min')

Final shape: 2,823,044 rows  x  54 columns

=== Join Coverage ===
  NOAA weather      :  2,697,299  (95.5%)
  T-100 capacity    :  2,822,647  (100.0%)
  DB1B fare         :  2,644,388  (93.7%)

=== Null Rates for Key Features ===
                    Null Count  Null %
wx_had_ts               125745     4.5
wx_min_vis              125745     4.5
wx_max_wind             125745     4.5
wx_total_prcp           125745     4.5
t100_seats                 397     0.0
t100_carrier_share         423     0.0
db1b_avg_fare           178656     6.3
dep_hour_bucket              0     0.0
season                       0     0.0
is_weekend                   0     0.0
is_holiday_week              0     0.0
ArrDelayMinutes          75040     2.7
NASDelay               2280785    80.8
WeatherDelay           2280785    80.8
is_late                      0     0.0

=== Delay Target (operated flights only) ===
  On-time (<= 15 min): 80.8%
  Late    (>  15 min): 19.0%
  Mean arrival delay:  14.2 min
  Median a

## Section 8 — Save

Write the joined, feature-enriched dataset. The internal `date` column (Python date objects)
is dropped — `FlightDate` (datetime64) covers the same information.

In [9]:
ot_out = ot.drop(columns=['date'])

out_path = PROJECT_ROOT / PROCESSED_DATA_PATH / 'flights_featured.parquet'
ot_out.to_parquet(out_path, index=False, engine='pyarrow')

size_mb = out_path.stat().st_size / 1e6
print(f'Saved:  {out_path.name}')
print(f'Size:   {size_mb:.1f} MB')
print(f'Shape:  {ot_out.shape[0]:,} rows  x  {ot_out.shape[1]} columns')
print()
print('Column schema:')
for col in ot_out.columns:
    dtype    = str(ot_out[col].dtype)
    null_pct = round(ot_out[col].isna().mean() * 100, 1)
    print(f'  {col:<38} {dtype:<14}  {null_pct:>5.1f}% null')

Saved:  flights_featured.parquet
Size:   64.4 MB
Shape:  2,823,044 rows  x  53 columns

Column schema:
  Month                                  int64             0.0% null
  DayOfWeek                              int64             0.0% null
  FlightDate                             datetime64[ns]    0.0% null
  Reporting_Airline                      object            0.0% null
  Flight_Number_Reporting_Airline        float64           0.0% null
  Origin                                 object            0.0% null
  OriginCityName                         object            0.0% null
  OriginStateName                        object            0.0% null
  Dest                                   object            0.0% null
  DestCityName                           object            0.0% null
  DestState                              object            0.0% null
  CRSDepTime                             int64             0.0% null
  DepTime                                float64           2.4% null
